# Concurrent Requests with Python Asyncio TaskGroup and Data Library for Python

## What is Python Asyncio? And how it relate to Data Library for Python

Python [asyncio](https://docs.python.org/3/library/asyncio.html) is a library for writing concurrent code with `async/await`. Common asyncio interfaces for concurrent programming include:

- [`asyncio.create_task(coro)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.create_task), which wraps *one [coroutine](https://docs.python.org/3/library/asyncio-task.html#coroutine)* in a [Task](https://docs.python.org/3/library/asyncio-task.html#asyncio.Task) and schedules it for execution, returning the Task object.
- [`asyncio.gather(*aws)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather), which runs [awaitables](https://docs.python.org/3/library/asyncio-task.html#asyncio-awaitables) in the `aws` sequence concurrently.
- [Task Groups](https://docs.python.org/3/library/asyncio-task.html#task-groups), a modern task-management API that provides a reliable way to wait for all tasks in a group to finish using [`asyncio.TaskGroup`](https://docs.python.org/3/library/asyncio-task.html#asyncio.TaskGroup) together with task creation APIs such as [`asyncio.create_task(coro)`](https://docs.python.org/3/library/asyncio-task.html#asyncio.create_task).

This notebook is a sequel to my [Data Library for Python with Asyncio Gather](./ld_notebook_async_gather.ipynb) example notebook. It demonstrates how to use the [Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python) Content Layer Historical Pricing `get_data_async` method for requesting multiple RICs concurrently using the modern (Python 3.11) asyncio`TaskGroup` interface.

This Jupyter Notebook uses a *Platform Session* connection. If you are using another session type, please refer to the [Data Library Quickstart](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python/quick-start) for more details.

## Prerequisite  

The basic knowledge of Python Asyncio library is required before proceeding. You can learn the basic of asyncio from the following resources:

- [Python's asyncio: A Hands-On Walkthrough](https://realpython.com/async-io-python/)
- [Asyncio Architecture in Python: Event Loops, Tasks, and Futures Explained](https://dev.to/imsushant12/asyncio-architecture-in-python-event-loops-tasks-and-futures-explained-4pn3)
- [A Conceptual Overview of asyncio](https://docs.python.org/3/howto/a-conceptual-overview-of-asyncio.html)
- [Python Asynchronous I/O](https://docs.python.org/3/library/asyncio.html)

The others are the Data Platform account (Machine-ID type), Python, and Data Library for Python.

### Do I need to understand Asyncio.Gather before proceeding this example?

No, you don't. However, if you understand both approaches, you can choose which one is better for your application and business requirements. 

Now we come to the Data Library code part, the first step is importing all requires libraries

In [1]:
import os
import asyncio
from IPython.display import Markdown, display
import lseg.data as ld
from lseg.data import session
from lseg.data.content import historical_pricing
from lseg.data._errors import LDError
from lseg.data.content.historical_pricing import Adjustments, Intervals
from dotenv import load_dotenv
import pandas as pd
pd.set_option("future.no_silent_downcasting", True)

Next, use the [python-dotenv](https://pypi.org/project/python-dotenv/) library to get the Data Platform credential from the `.env` file.

**Note**: The `.env` file **should not be committed** to the version control repository.

In [2]:
# Load environment variables from .env file
load_dotenv(dotenv_path='.env')
# Retrieve Platform Session credentials from environment variables
app_key = os.getenv('LSEG_API_KEY')
username = os.getenv('LSEG_MACHINE_ID')
password = os.getenv('LSEG_PASSWORD')

Then, we create a library `session` object to manage an application session. The session automatic define authentication, manage connection resources, and expose the APIs to retrieve data for an application.

In [3]:
# Create a platform session with provided credentials for authentication
ld_session = session.platform.Definition(
         app_key=app_key,
         grant=session.platform.GrantPassword(
             username=username,
             password=password
         ),
         signon_control=True
).get_session()

# Set this session as the default for all subsequent Data Library calls
session.set_default(ld_session)

# Open the connection to the LSEG Data Platform
ld_session.open()

<OpenState.Opened: 'Opened'>

The session is opened, we can declare our historical data related variable.

In [4]:
# ── Instrument universe ────────────────────────────────────────────────────────
INSTRUMENTS = {
    "NVDA.O":  "NVIDIA",
    "AAPL.O":  "Apple",
    "MSFT.O":  "Microsoft",
    "AMZN.O":  "Amazon",
    "GOOG.O":  "Alphabet",
    "AVGO.O":  "Broadcom",
    "META.O":  "Meta",
    "ORCL.N":  "Oracle",
    "IBM.N":   "IBM",
    "PLTR.O":  "Palantir",
    "NFLX.O":  "Netflix",
    "TSLA.O":  "Tesla",
    "CRM.N":   "Salesforce",
    "AMD.O":   "AMD",
    "INTC.O":  "Intel",
    "ARM.O":   "Arm Holdings",
    "TXN.O": "Texas Instruments",
    "CSCO.O":  "Cisco Systems",
    "WMT.O":   "Walmart",
    "LLY.N":   "Eli Lilly and Company",
    "JPM.N":   "JPMorgan Chase & Co.",
    "XOM.N":   "Exxon Mobil Corporation",
    "V.N":     "Visa Inc.",
    "JNJ.N":   "Johnson & Johnson",
    "MU.O":    "Micron Technology, Inc.",
    "MA.N":    "Mastercard Incorporated",
    "COST.O":  "Costco Wholesale Corporation",
    "CVX.N":   "Chevron Corporation",
    "BAC.N":   "Bank of America Corporation",
    "CAT.N":   "Caterpillar Inc.",
}

# ── Date range ─────────────────────────────────────────────────────────────────
START = "2025-12-01T00:00:00Z"
END   = "2026-06-30T23:59:59Z"

# ── Event correction filters ───────────────────────────────────────────────────
# Only include exchange-level and manual corrections; filters out duplicates
# and administrative adjustments that would distort event counts.
EVENT_ADJUSTMENTS = [Adjustments.EXCHANGE_CORRECTION, Adjustments.MANUAL_CORRECTION]

# ── Field lists ─────────────────────────────────────────────────────────────────
# Defined once as constants so that each list comprehension in the next cell
# reuses the same object rather than allocating a new list on every iteration.
EVENT_FIELDS    = ["EVENT_TYPE", "TRDPRC_1", "TRDVOL_1"]
INTRADAY_FIELDS = ["TRDPRC_1", "BID", "ASK"]
INTERDAY_FIELDS = ["BID", "ASK", "OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "NUM_MOVES", "TRNOVR_UNS"]



## Data Library Access Layer `get_history` VS Content Layer `historical_pricing`

Let's start with why we need the Data Library Content Layer Historical Pricing module instead of just the basic `get_history` method. 

The `get_history` method is part of the Data Library *Access Layer**. The Access Layer provides the easiest way to get data via a set of simple API interfaces for developers.  While the `get_history` lets developers retrieve pricing history for both single and multiple instruments via a single function call. All Access Layer methods including the `get_history` are in synchronous execution which block other tasks, an application must wait until the function call is finished. 


The `historical_pricing` module is part of the *Content Layer*. The Content Layer allows developers to access the same content as Access Layer which are a more flexible manner:
- Richer / fuller response e.g. metadata, sentiment scores - where available.
- Using Asynchronous or Event-Driven operating modes - in addition to Synchronous.
- The layer interfaces are based on logical market data objects such as Level 1 Market Price Data (snapshot/streaming), News, Historical Pricing, Bond Analytics, Environmental & Social Governance (ESG), Search, IPA, and so on.

The module lets developers set historical data query via *definition* then get data via synchronous `get_data` and asynchronous `get_data_async` methods. I am focusing on the asynchronous `get_data_async` method of the Historical Pricing module here.

## Using Asyncio TaskGroup

Introduced in Python 3.11, the [Task Groups](https://docs.python.org/3/library/asyncio-task.html#task-groups) provides a modern and reliable way to manage multiple concurrent tasks using [`asyncio.TaskGroup`](https://docs.python.org/3/library/asyncio-task.html#asyncio.TaskGroup) interface. It supports structured concurrency, which means tasks created inside the group are managed together and automatically awaited when the group exits

Although the TaskGroup lets developers *run and get results for many tasks* similar to `asyncio.gather` method. However,  it behaves differently in several important ways.

### TaskGroup must be used with Python async with statement

To use Asyncio TaskGroup, developers must create tasks inside an the the asynchronous context manager using `async with` statement and the [`TaskGroup.crate_task`](https://docs.python.org/3/library/asyncio-task.html#asyncio.TaskGroup.create_task) method. The group automatically waits for all tasks to finish when the async with block exits.

Example Code:

```python
async def main():
    async with asyncio.TaskGroup() as tg:
        task1 = tg.create_task(some_coro(...))
        task2 = tg.create_task(another_coro(...))
    print(f"Both tasks have completed now: {task1.result()}, {task2.result()}")
```

The `async with` block implicitly waiting for all tasks when the block exits. While the block is still active, new tasks can still be added to the same task group. Once the last task finishes and the `async with` block is exited, no new tasks may be added to the group. 

### Getting Result

Unlike the `asyncio.gather` that returns a list of results in the same order as input, TaskGroup does not directly return a list of results. Developers must keep references to the tasks and call `.result()` method after the async with block complete as an example above.

The completion, return results and execution order are definitely **not guaranteed** event if the coroutines are created sequentially, the actual network operations, I/O, OS, etc.

### Failure Handling

This is the most important difference between `TaskGroup` and `asyncio.gather()`. If one task in the TaskGroup fails, the group automatic **cancels the remaining tasks and raises an ExceptionGroup**. This **Fails Fast** behavior helps prevent orphaned background tasks and reduces the risk of resource leaks . 


TaskGroup is a good choice all tasks in the same operation and should succeed or fail together. The async with block also helps ensure that task cleanup is handled properly before execution continues.

The first step is defining a new `display_response_taskgroup` method that supports TaskGroup Fails Fast behavior.

In [5]:
def display_response_taskgroup(data):
    """Display results from asyncio.TaskGroup tasks.

    TaskGroup raises an ExceptionGroup if any task fails, so by the time
    results reach this function every item is a successful API response.
    """
    for idx, api_response in enumerate(data, start=1):
        closure = getattr(api_response, "closure", f"result-{idx}")
        print(f"\n[{idx}] Response received for: {closure}")

        if api_response.is_success:
            display(api_response.data.df)
        else:
            # HTTP-level failure (4xx / 5xx from the platform).
            print(f"Request failed - HTTP status: {api_response.http_status}")

You see that it is less complicated than the `display_response` helper function with `asyncio.gather(..., return_exceptions=True)` approach.

### Requesting Data with Asyncio TaskGroup, The Basic Form

I am demonstrating the most basic form of grouping multiple calls via Asyncio TaskGroup using  for Historical Pricing Interday data.

- All requests are created as tasks inside the `async with asyncio.TaskGroup()` block, which runs them concurrently and waits until every one finishes before moving on.
- Each task carries a `closure` label (the company name) so you can identify which response belongs to which instrument when displaying results.

In [6]:
# Convert dictionary items to (RIC, company) pairs so each request can carry a readable label.
list_of_games_companies = list({"TTWO.O": "Take-Two Interactive","EA.O": "Electronic Arts"} .items())

# Run one monthly-summary request per instrument at the same time.
# TaskGroup waits for all tasks to finish before moving on.
try:
    async with asyncio.TaskGroup() as tg:
        tasks = [
            tg.create_task(
                historical_pricing.summaries.Definition(
                    universe=ric,              # RIC symbol, e.g. "GOOG.O"
                    fields=INTERDAY_FIELDS,
                    interval=Intervals.MONTHLY,
                    start=START,
                    end=END,
                    adjustments=[
                        Adjustments.EXCHANGE_CORRECTION, 
                        Adjustments.MANUAL_CORRECTION,
                        Adjustments.CCH,
                        Adjustments.CRE,
                        Adjustments.RPO,
                        Adjustments.RTS
                    ],                 
                    count=10           # Max number of records to return; adjust as needed
                ).get_data_async(
                    closure=f"{company}"
                )
            )
            for ric, company in list_of_games_companies
        ]
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

After the `TaskGroup` block finishes, the tasks are complete but their results have not been collected yet.

The cell below does two things:
1. Calls `task.result()` on every task to extract the response data into `historical_data`, preserving the same order as the original instrument list.
2. Displays each response as a DataFrame using `display_response_taskgroup`.

In [7]:
try:
    # Collect completed responses in the same order as 'instruments'.
    historical_data = [task.result() for task in tasks]

    display(Markdown("**Game Companies Historical Price Interday Historical Summaries (Monthly) using TaskGroup**"))
    display_response_taskgroup(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Game Companies Historical Price Interday Historical Summaries (Monthly) using TaskGroup**


[1] Response received for: Take-Two Interactive


TTWO.O,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2025-12-31,256.02,256.08,246.07,260.04,240.28,256.03,734662,7339159660
2026-01-31,220.2,220.35,257.705,258.56,211.13,220.3,898518,8139970064
2026-02-28,211.42,211.5,225.51,229.075,188.65,211.48,1368267,12504713533
2026-03-31,197.42,197.47,209.002,221.08,187.63,197.5,1039320,8935297052
2026-04-30,213.77,213.87,198.0,221.22,193.79,213.76,883854,6325025317
2026-05-31,224.2,224.23,215.46,247.0,211.5845,224.16,1305113,12281388257
2026-06-30,249.81,249.96,227.99,252.95,206.0,249.98,1540226,14775265605



[2] Response received for: Electronic Arts


EA.O,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2025-12-31,204.33,204.44,201.99,204.885,201.7,204.33,571018,8609979458
2026-01-31,203.9,203.92,204.4,204.72,203.05,203.92,585527,8003853048
2026-02-28,200.55,200.56,203.81,204.15,196.4,200.57,885255,12855532650
2026-03-31,203.76,203.79,200.25,203.9,197.01,203.87,750224,11103818938
2026-04-30,202.36,202.37,203.35,204.22,201.78,202.37,549112,6568497385
2026-05-31,201.72,201.77,202.69,202.69,199.45,201.72,626262,8112181998
2026-06-30,204.99,205.01,201.93,205.6899,201.54,205.04,718790,8976535883


Please be noticed that when developers send multiple Historical Pricing Definition with **a single RIC** request, each RIC gets its own data response grouping together sequently in a Python *list* returns from `historical_data = [task.result() for task in tasks]` statement.

In [8]:
print(f" Data type is {type(historical_data)} and length is {len(historical_data)}")

 Data type is <class 'list'> and length is 2


An application can iterate each result from the returned list (as shown in the `display_response_taskgroup` method) or use the following statement to get data from its closure.

In [9]:
next(
    response.data.df
    for response in historical_data
    if getattr(response, "closure", None) == "Take-Two Interactive"
 )

TTWO.O,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2025-12-31,256.02,256.08,246.07,260.04,240.28,256.03,734662,7339159660
2026-01-31,220.2,220.35,257.705,258.56,211.13,220.3,898518,8139970064
2026-02-28,211.42,211.5,225.51,229.075,188.65,211.48,1368267,12504713533
2026-03-31,197.42,197.47,209.002,221.08,187.63,197.5,1039320,8935297052
2026-04-30,213.77,213.87,198.0,221.22,193.79,213.76,883854,6325025317
2026-05-31,224.2,224.23,215.46,247.0,211.5845,224.16,1305113,12281388257
2026-06-30,249.81,249.96,227.99,252.95,206.0,249.98,1540226,14775265605


### Understanding asyncio.TaskGroup Code

The code above shows a basic example of using `asyncio.TaskGroup` to run multiple Historical Pricing module `.get_data_async(...)` requests concurrently in one managed block.

In this notebook, each `(ric, company)` request is one task. Running them together is usually faster than running them one-by-one.

#### Why use TaskGroup

`TaskGroup` provides structured concurrency: tasks are created inside a single managed `async with` block and are awaited together when the block exits.

This is a strong fit for strict batch workflows. If one task fails, `TaskGroup` cancels unfinished sibling tasks and raises the failures as one `ExceptionGroup`.

#### TaskGroup vs asyncio.gather (easy view)

- `asyncio.gather(...)` runs many awaitables and returns a result list.
- `asyncio.TaskGroup` creates tasks inside `async with` and waits for the group to finish automatically.
- Biggest behavior difference:
  - `gather` is flexible when you want partial results.
  - `TaskGroup` is fail-fast: if one task fails, remaining tasks are cancelled.

#### What this code is doing

1. Create one async request per `(ric, company)` pair using `historical_pricing.summaries.Definition(...).get_data_async(...)`.
2. Add each coroutine to the group with `tg.create_task(...)`.
3. Let `async with asyncio.TaskGroup()` wait until all tasks complete (or are cancelled).
4. After the block exits, every task is either finished or cancelled due to an error.

After the `TaskGroup` block exits, use this statement to read the returned data.

```python
historical_data = [task.result() for task in tasks]
```

#### Error handling (simple)

- Keep the TaskGroup block inside `try/except* LDError`.
- Call `task.result()` only after the TaskGroup block exits.
- The helper `display_response_taskgroup(...)` methods show each response safely with defensive display pattern.

#### How to get data for one instrument

Each async request uses `closure=f"{company}"`, so every response carries an instrument label.

To find one instrument, match the response `closure` string (for example, company name like the code that get `"Take-Two Interactive"` above) when reviewing outputs.

### Can I use Historical Pricing Events and Summaries Definitions in the same Asyncio TaskGroup?

Yes, you can. The cell below shows how.

The idea is to prepare two separate groups of requests first:

- **Event requests** - these fetch recent trade and correction events for Boeing (*BA.N*).
- **Intraday summary requests** - these fetch 5-minute price bars over the date range for Lockheed Martin (*LTM.N*).

Both groups are built as lists of coroutines (pending async calls, not yet running). They are then merged into a single list called `all_coroutines`, keeping their original order.

Separating the setup from the execution makes the code easier to read and lets you confirm what will be requested before anything runs.

In [10]:
# ── Event tasks (BA.N: Boeing) ─────────────────────────────────────────────
# Fetch the 10 most recent trade/correction events per instrument concurrently.
event_tasks = [
    historical_pricing.events.Definition(
            universe="BA.N",
            fields=EVENT_FIELDS,
            adjustments=EVENT_ADJUSTMENTS,
            count=10,
        ).get_data_async(
            closure="Boeing Historical Events"
        )
]

# ── Intraday summary tasks (LTM.N: Lockheed Martin) ───────────────────────────────────────
# Fetch 5-minute OHLC bars for the date range for the next 10 instruments.
intraday_summary_tasks = [
     historical_pricing.summaries.Definition(
            universe="LTM.N",
            fields=INTRADAY_FIELDS,
            interval=Intervals.FIVE_MINUTES,
            count=10
        ).get_data_async(
            closure=f"Lockheed Martin Historical Summaries ({Intervals.FIVE_MINUTES})"
        )
]

all_coroutines = [*event_tasks, *intraday_summary_tasks]

Once `all_coroutines` list is ready, the next cell runs everything at the same time using `asyncio.TaskGroup`.

- Unlike `asyncio.gather`, where you can pass mixed Data Library definitions to awaitables directly, `TaskGroup` works best when we first combine Event and Intraday Summary calls into one `all_coroutines` list and then create tasks from that list in a single run.
- Keeping them in one ordered list makes result handling easier: the first part of `historical_data` maps to Event results, and the second part maps to Intraday Summary results.

In [11]:
# Run all requests concurrently.
# TaskGroup waits for every task before exiting the block.
try:
    # Execute all request coroutines concurrently with TaskGroup (Python 3.11+).
    # Structured concurrency ensures tasks finish (or are cancelled together)
    # before execution continues past the async-with block.
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(coro) for coro in all_coroutines]

    # Keep results in task-creation order so [0:5] and [5:] map cleanly
    # to the event and intraday request groups.
    historical_data = [task.result() for task in tasks]
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

Then we can display the Historical data as follows:

In [12]:
display_response_taskgroup(historical_data)


[1] Response received for: Boeing Historical Events


BA.N,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-07-15 20:00:04.260,trade,218.27,1
2026-07-15 20:00:04.260,trade,218.27,916
2026-07-15 20:00:04.260,trade,218.2,1200
2026-07-15 20:00:04.318,trade,218.29,458
2026-07-15 20:00:04.363,trade,218.12,630793
2026-07-15 20:00:04.363,trade,218.29,300
2026-07-15 20:00:06.110,trade,218.12,<NA>
2026-07-15 20:10:00.001,trade,218.12,<NA>
2026-07-15 22:30:00.002,trade,218.12,<NA>



[2] Response received for: Lockheed Martin Historical Summaries (PT5M)


LTM.N,TRDPRC_1,BID,ASK
Timestamp,,,
2026-07-15 19:30:00,54.56,54.54,54.61
2026-07-15 19:35:00,54.54,54.53,54.6
2026-07-15 19:40:00,54.66,54.61,54.71
2026-07-15 19:45:00,54.64,54.49,54.81
2026-07-15 19:50:00,54.99,54.9,55.02
2026-07-15 19:55:00,54.74,54.74,54.88
2026-07-15 20:00:00,54.87,54.7,54.91
2026-07-15 20:10:00,54.87,<NA>,<NA>
2026-07-15 22:30:00,54.87,<NA>,<NA>


### Requesting Data with Request Limits

Like `asyncio.gather()`, we can use `asyncio.TaskGroup` with use [`asyncio.Semaphore`](https://docs.python.org/3/library/asyncio-sync.html#asyncio.Semaphore) to limit the number of in-flight requests (that we are grouping) at any given time (default: 3).

If you are requesting only 2-10 RICs, the backend can usually handle the load without issue. As the number of simultaneous requests grows to 50, 100, or more, a semaphore becomes essential for staying within the platform's rate limits (see the **Rate Limit** section in the README). The following example demonstrates this pattern with 20 RICs.

The semaphore pattern is recommended when requesting a large number of instruments from the platform.

In [ ]:
# ── Demo universe for semaphore-throttled TaskGroup requests ─────────────────
artemis_ii = {
    "LTM.N": "Lockheed Martin",
    "BA.N": "Boeing",
    "NOC.N": "Northrop Grumman",
    "LHX.N": "L3Harris Technologies",
    "AIR.PA": "Airbus",
}

# Convert dictionary items to (RIC, company) pairs so each request can carry a readable label.
list_of_artemis_ii_companies = list(artemis_ii.items())

# Keep only 2 in-flight requests at a time to demonstrate throttling.
semaphore = asyncio.Semaphore(2)

async def fetch_summaries_throttle(ric, company, fields, interval,count):
    """Request historical summaries for one RIC with semaphore throttling."""
    # Semaphore enforces max concurrent requests across all created coroutines.
    async with semaphore:
        return await historical_pricing.summaries.Definition(
            universe=ric,
            fields=fields,
            interval=Intervals.MONTHLY,
            start=START,
            end=END,
            count=count,
        ).get_data_async(
            closure=f"{company} Historical Summaries ({interval})"
        )

# Build request coroutines; TaskGroup in the next cell schedules them concurrently.
interaday_summary_tasks = [
    fetch_summaries_throttle(ric, company, INTERDAY_FIELDS, Intervals.MONTHLY, count=5)
    for ric, company in list_of_artemis_ii_companies
]

Once the semaphore-protected coroutines are ready, the next step is to schedule them as tasks using `asyncio.TaskGroup` as I have demonstrated previously.

In [19]:
# Run all requests concurrently.
# TaskGroup waits for every task before exiting the block.
try:
    # Execute all request coroutines concurrently with TaskGroup (Python 3.11+).
    # Structured concurrency ensures tasks finish (or are cancelled together)
    # before execution continues past the async-with block.
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(coro) for coro in interaday_summary_tasks]

    # Keep results in task-creation order cleanly
    # to the event and intraday request groups.
    historical_data = [task.result() for task in tasks]
    display(Markdown("**Artemis II involved Companies Historical Price Events using TaskGroup and Semaphore**"))
    display_response_taskgroup(historical_data)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Artemis II involved Companies Historical Price Events using TaskGroup and Semaphore**


[1] Response received for: Lockheed Martin Historical Summaries (P1M)


LTM.N,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-02-28,56.1,56.15,65.66,70.09,55.98,56.16,53632,367440890
2026-03-31,49.44,49.54,53.59,53.795,45.61,49.44,56529,330304932
2026-04-30,47.53,47.57,50.42,57.04,46.71,47.53,42591,230454941
2026-05-31,53.65,53.75,47.65,54.09,45.12,53.68,38507,201970719
2026-06-30,58.31,58.35,53.63,58.9,46.64,58.27,32276,197517292



[2] Response received for: Boeing Historical Summaries (P1M)


BA.N,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-02-28,227.49,227.53,232.64,247.67,224.93,227.53,199752,5780747672
2026-03-31,198.98,199.07,225.25,231.96,187.81,199.03,281572,6800293764
2026-04-30,228.95,229.02,204.2,236.41,203.0,229.03,242018,5535802305
2026-05-31,231.06,231.07,230.73,244.0,214.02,231.15,268649,6780865118
2026-06-30,216.4,216.54,228.67,231.33,208.0,216.47,254528,7572568404



[3] Response received for: Northrop Grumman Historical Summaries (P1M)


NOC.N,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-02-28,724.71,724.94,685.0,745.0,674.88,724.38,73960,3136368627
2026-03-31,682.0,682.12,745.0,773.69,663.93,682.24,84954,3892715734
2026-04-30,579.39,579.62,689.34,708.55,567.34,579.48,81736,2786895972
2026-05-31,563.61,563.68,580.0,580.0,539.35,563.68,82971,2834635997
2026-06-30,508.96,509.3,554.95,559.65,493.945,509.31,117621,3759077849



[4] Response received for: L3Harris Technologies Historical Summaries (P1M)


LHX.N,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-02-28,364.54,364.68,338.0,365.68,329.89,364.54,86570,2216586727
2026-03-31,345.13,345.17,370.32,379.23,337.14,345.15,114789,3273772749
2026-04-30,320.63,320.82,348.84,364.21,313.01,320.55,108919,2555885320
2026-05-31,315.03,315.04,322.0,322.0,296.83,315.18,98083,2468076624
2026-06-30,290.32,290.43,311.18,318.43,282.48,290.59,114796,2949656753



[5] Response received for: Airbus Historical Summaries (P1M)


AIR.PA,BID,ASK,OPEN_PRC,HIGH_1,LOW_1,TRDPRC_1,NUM_MOVES,TRNOVR_UNS
Date,,,,,,,,
2026-02-28,184.22,184.5,189.4,201.2,180.18,184.24,370707,4626720808.35
2026-03-31,160.82,160.88,179.96,182.14,157.42,160.82,569029,5746982023.55
2026-04-30,174.86,174.88,167.82,181.96,160.82,174.88,507672,4908850531.94
2026-05-31,179.64,179.7,174.74,191.92,164.74,179.7,452423,5072124021.01
2026-06-30,194.52,194.54,177.3,196.9,167.9,194.54,434281,4933407027.6


### Understanding asyncio.TaskGroup with asyncio.Semaphore

The code above uses `asyncio.TaskGroup` with `asyncio.Semaphore` to run many requests safely.

#### Where Semaphore fits

If you start too many requests at once, you may hit HTTP **429 Too Many Requests**.
A semaphore prevents that by limiting how many requests are in-flight at the same time.

#### How it works

Only *N* coroutines can enter the protected section at once. The rest wait.

```
asyncio.Semaphore(N)  ->  at most N HTTP requests in-flight at any time
```

#### Key points

- Create the semaphore once, outside the helper function.
- Put `await get_data_async(...)` inside `async with semaphore:`.
- The semaphore limits request concurrency per coroutine, while `TaskGroup` creates and waits for all tasks.
- Start with a small limit (for example 5-10) and lower it if you still see 429 errors.

#### What `fetch_summaries_throttle` helper method does

The method uses `historical_pricing.summaries.Definition(...)` to request summary bars for one instrument. The key important thing is the method runs inside `async with semaphore:` so request concurrency is limited for each coroutine.

Semaphore controls how many requests run at once; TaskGroup manages the task lifecycle.

### Can I use different Definitions in the same Asyncio TaskGroup with a Semaphore?

Yes. The only additional step is to wrap each request in `async with semaphore` through a helper function; the rest of the TaskGroup pattern remains the same.

In [20]:
REQUEST_LIMITS = 5
semaphore = asyncio.Semaphore(REQUEST_LIMITS)

# Convert dictionary items to (RIC, company) pairs so each request can carry a readable label.
list_of_sp500_companies = list(INSTRUMENTS.items())

async def fetch_events_throttle(ric, company, fields, count):
    """Request event data for one RIC with semaphore throttling."""
    async with semaphore:
        return await historical_pricing.events.Definition(
            universe=ric,
            fields=fields,
            adjustments=EVENT_ADJUSTMENTS,
            count=count,
        ).get_data_async(
            closure=f"{company} Historical Events"
        )

# ── Event tasks (INSTRUMENTS[0:10]) ─────────────────────────────────────────────
# Fetch the 10 most recent trade/correction events per instrument concurrently.
event_tasks = [
    fetch_events_throttle(ric, company, EVENT_FIELDS, 10)
    for ric, company in list_of_sp500_companies[:10]
]

# ── Intraday summary tasks (INSTRUMENTS[20:30]) ───────────────────────────────────────
# Fetch 5-minute OHLC bars for the date range for the next 10 instruments.
intraday_summary_tasks = [
    fetch_summaries_throttle(ric, company, INTRADAY_FIELDS, Intervals.FIVE_MINUTES, count=5)
    for ric, company in list_of_sp500_companies[20:30]
]

# ── Flatten all coroutines into a single ordered list ──────────────────────────────
# The next cell executes them concurrently via asyncio.TaskGroup (Python 3.11+).
all_coroutines = [*event_tasks, *intraday_summary_tasks]

#### What `fetch_summaries_throttle` helper method does?

The method uses `historical_pricing.events.Definition(...)` to request event-level data for one instrument. Like the `fetch_summaries_throttle` method above, it also runs inside `async with semaphore:` so request concurrency is limited for each coroutine.

In [21]:
# Run all requests concurrently.
# TaskGroup waits for every task before exiting the block.
try:
    # Execute all request coroutines concurrently with TaskGroup (Python 3.11+).
    # Structured concurrency ensures tasks finish (or are cancelled together)
    # before execution continues past the async-with block.
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(coro) for coro in all_coroutines]

    # Keep results in task-creation order so [0:5] and [5:] map cleanly
    # to the event and intraday request groups.
    historical_data = [task.result() for task in tasks]
    display(Markdown("**Companies Historical Price Events using TaskGroup**"))
    display_response_taskgroup(historical_data[:10])
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Historical Price Events using TaskGroup**


[1] Response received for: NVIDIA Historical Events


NVDA.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-07-09 23:59:55.636,trade,202.28,30.0
2026-07-09 23:59:56.029,trade,202.2891,1.0
2026-07-09 23:59:56.228,trade,202.334,50.0
2026-07-09 23:59:56.689,trade,202.2817,0.044905
2026-07-09 23:59:57.858,trade,202.3,10.0
2026-07-09 23:59:57.879,trade,202.3,20.0
2026-07-09 23:59:57.929,trade,202.28,30.0
2026-07-09 23:59:58.883,trade,202.33,154.0
2026-07-09 23:59:58.948,trade,202.35,49.0



[2] Response received for: Apple Historical Events


AAPL.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-07-09 23:59:29.588,trade,315.2801,156
2026-07-09 23:59:30.572,trade,315.2,4
2026-07-09 23:59:57.789,trade,315.31,5
2026-07-09 23:59:57.789,trade,315.31,14
2026-07-09 23:59:57.789,trade,315.31,1
2026-07-09 23:59:57.858,trade,315.3,1
2026-07-09 23:59:57.858,trade,315.31,9
2026-07-09 23:59:57.858,trade,315.31,1
2026-07-09 23:59:57.904,trade,315.21,20



[3] Response received for: Microsoft Historical Events


... (11 outputs left out)


[9] Response received for: IBM Historical Events


IBM.N,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-07-09 20:00:03.134,trade,295.14,4
2026-07-09 20:00:03.134,trade,295.19,10
2026-07-09 20:00:03.134,trade,295.2,20
2026-07-09 20:00:03.134,trade,295.2,247
2026-07-09 20:00:03.134,trade,295.2,200
2026-07-09 20:00:03.184,trade,295.3,1091490
2026-07-09 20:00:06.133,trade,295.3,<NA>
2026-07-09 20:10:00.003,trade,295.3,<NA>
2026-07-09 22:30:00.003,trade,295.3,<NA>



[10] Response received for: Palantir Historical Events


PLTR.O,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-07-09 23:59:41.554,trade,128.7999,100
2026-07-09 23:59:42.912,trade,128.8,2
2026-07-09 23:59:43.436,trade,128.79,1
2026-07-09 23:59:46.355,trade,128.8,1
2026-07-09 23:59:46.606,trade,128.79,1
2026-07-09 23:59:46.857,trade,128.8,1
2026-07-09 23:59:47.109,trade,128.8,1
2026-07-09 23:59:55.002,trade,128.7655,50
2026-07-09 23:59:57.894,trade,128.76,22


In [22]:
try:
    display(Markdown("**Companies Intraday Daily Historical Summaries (5-Minute Intervals) using TaskGroup**"))
    display_response_taskgroup(historical_data[10:])
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Intraday Daily Historical Summaries (5-Minute Intervals) using TaskGroup**


[1] Response received for: JPMorgan Chase & Co. Historical Summaries (PT5M)


JPM.N,TRDPRC_1,BID,ASK
Date,,,
2026-02-28,300.3,299.92,300.12
2026-03-31,294.16,294.16,294.27
2026-04-30,313.23,313.27,313.31
2026-05-31,299.31,299.42,299.43
2026-06-30,327.33,327.2,327.35



[2] Response received for: Exxon Mobil Corporation Historical Summaries (PT5M)


XOM.N,TRDPRC_1,BID,ASK
Date,,,
2026-02-28,152.5,152.6,152.61
2026-03-31,169.66,169.63,169.67
2026-04-30,154.33,154.34,154.45
2026-05-31,145.26,145.35,145.36
2026-06-30,136.72,136.44,136.49



[3] Response received for: Visa Inc. Historical Summaries (PT5M)


... (11 outputs left out)


[9] Response received for: Bank of America Corporation Historical Summaries (PT5M)


BAC.N,TRDPRC_1,BID,ASK
Date,,,
2026-02-28,49.83,49.8,49.81
2026-03-31,48.75,48.74,48.75
2026-04-30,53.46,53.47,53.48
2026-05-31,51.6,51.6,51.61
2026-06-30,56.98,56.95,56.96



[10] Response received for: Caterpillar Inc. Historical Summaries (PT5M)


CAT.N,TRDPRC_1,BID,ASK
Date,,,
2026-02-28,742.83,742.84,743.14
2026-03-31,708.46,708.05,708.4
2026-04-30,890.11,890.1,890.75
2026-05-31,875.87,875.6,875.75
2026-06-30,1064.9,1064.46,1064.6


### How Does TaskGroup Handle Errors?

`asyncio.TaskGroup` uses a **fail-fast** approach. If one task raises an error, TaskGroup cancels the remaining tasks and raises an `ExceptionGroup`. This means the whole batch operation fails as a unit instead of returning partial results to the application.

**Note**: This error-handling example uses only a small number of RICs, so `asyncio.Semaphore` is not required in this section.

#### Invalid and Non-Permission RIC Requests

The example below demonstrates two error cases:

- an invalid RIC: `INVALIDRIC.O`
- a non-permission RIC: `ASML.AS` (permission may vary by user account)

This shows how TaskGroup stops the overall operation when one or more requests fail.

In [23]:
invalid_instrument_cases = {"IBM.N":"IBM","INVALIDRIC.O": "Invalid Instrument","JPM.N":"JPMorgan Chase & Co.","ASML.AS": "ASML"}

# Convert dictionary items to (RIC, company) pairs so each request can carry a readable label.
list_of_rics_companies_with_errors = list(invalid_instrument_cases.items())
data_with_ric_errors = []

# Run one daily-summary request per instrument at the same time.
# TaskGroup waits for all tasks to finish before moving on.
try:
    async with asyncio.TaskGroup() as tg:
        tasks = [
            tg.create_task(
                historical_pricing.summaries.Definition(
                    universe=ric,              # RIC symbol, e.g. "GOOG.O"
                    fields=INTERDAY_FIELDS,
                    interval=Intervals.DAILY,
                    start=START,
                    end=END,
                    count=10             # Max number of records to return; adjust as needed
                ).get_data_async(
                    closure=f"{company} Interday Historical Summaries (Daily)"
                )
            )
            for ric, company in list_of_rics_companies_with_errors
        ]

    data_with_ric_errors = [task.result() for task in tasks]
    display(Markdown("**Companies Intraday Daily Historical Summaries (Daily) using TaskGroup with Errors**"))
    display_response_taskgroup(data_with_ric_errors)
except* LDError as errors:
    for error in errors.exceptions:
        print(f"Exception: {error}")

Exception: Error code TS.Interday.UserRequestError.70005 | The universe is not found.. Requested ric: INVALIDRIC.O. Requested fields: ['BID', 'ASK', 'OPEN_PRC', 'HIGH_1', 'LOW_1', 'TRDPRC_1', 'NUM_MOVES', 'TRNOVR_UNS']


As you can see, once the library encounters an error, it raises an exception and immediately marks the entire operation as failed. The error message is not added to the result list because TaskGroup raises the exception as soon as the failure occurs.

Please note that the execution order is **not guaranteed**, so you might get another error first when you run this code.

In [24]:
len(data_with_ric_errors)

0

#### Invalid Fields

Now let's see how the library handles invalid fields with the `asyncio.Taskgroup`.

In [25]:
EVENT_FIELDS_WITH_INVALID = EVENT_FIELDS + ["INVALID_FIELD"]  # Invalid field to demonstrate error handling
try:
    # Create a concurrent batch of event requests for the first three instruments.
    # The star-unpack passes each coroutine as a separate argument to gather.

    async with asyncio.TaskGroup() as tg:
        tasks = [
            tg.create_task(
                historical_pricing.events.Definition(universe="VOD.L", fields=EVENT_FIELDS_WITH_INVALID, count=5).get_data_async(closure="Vodaphone")
            )
        ]

    data_with_fields_error = [task.result() for task in tasks]

    # Display a section header before printing each response output.
    display(Markdown("**Companies Historical Price Events with RIC error**"))
    # Show each response DataFrame on success; otherwise print the HTTP status code.
    display_response_taskgroup(data_with_fields_error)
except* LDError as errors:
    for error in errors.exceptions:
        print(error)

**Companies Historical Price Events with RIC error**


[1] Response received for: Vodaphone


VOD.L,EVENT_TYPE,TRDPRC_1,TRDVOL_1
Timestamp,,,
2026-07-09 15:35:56.559,trade,97.76,379
2026-07-09 15:37:03.120,trade,97.76,648
2026-07-09 15:38:20.270,trade,97.76,150
2026-07-09 15:47:21.739,trade,97.3538,15633
2026-07-09 15:52:46.000,trade,97.76,2605


Like `asyncio.gather`, the library and backend can handle a mix of valid and invalid fields by ignoring the invalid ones and returning data for the valid fields.

The `asyncio.TaskGroup`, this mixed-valid-invalid fields request can still complete successfully.

You can use the `response.data.raw` statement to check what error that cause the fields are invalid.

In [26]:
data_with_fields_error[0].data.raw

{'universe': {'ric': 'VOD.L'},
 'adjustments': ['exchangeCorrection', 'manualCorrection'],
 'defaultPricingField': 'TRDPRC_1',
 'qos': {'timeliness': 'delayed'},
 'headers': [{'name': 'DATE_TIME', 'type': 'string'},
  {'name': 'EVENT_TYPE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRDVOL_1', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-07-09T15:52:46.000000000Z', 'trade', 97.76, 2605],
  ['2026-07-09T15:47:21.739000000Z', 'trade', 97.3538, 15633],
  ['2026-07-09T15:38:20.270000000Z', 'trade', 97.76, 150],
  ['2026-07-09T15:37:03.120000000Z', 'trade', 97.76, 648],
  ['2026-07-09T15:35:56.559000000Z', 'trade', 97.76, 379]],
 'status': {'code': 'TS.Intraday.UserRequestError.90006',
  'message': 'The universe does not support the following fields: [INVALID_FIELD].'},
 'meta': {'blendingEntry': {'headers': [{'name': 'COLLECT_DATETIME',
     'type': 'string'},
    {'name': 'RTL', 'type': 'number', 'decimalChar': '.'},
    {'name

**Please note that if a request contains only invalid fields (either one invalid field or a list of all invalid fields), the request fails and raises an exception.**

### Last Question: Can I make Asyncio TaskGroup return both successful results and errors?

In theory, yes—you can handle this programmatically. However, using `asyncio.gather(..., return_exceptions=True)` is usually much easier and keeps the code simpler for that requirement.

### Close the Session

Now we come to the last section of the code, you can close the session with the following statements.

In [27]:
# Close the session to release resources; in a long-running application, consider keeping the session open and reusing it for subsequent API calls instead.
ld_session.close()
ld.close_session()

## So, Which One Should I Use: Asyncio Gather or TaskGroup for Data Library?

There isn’t a single right answer. Both Asyncio Gather and TaskGroup are good options. The best choice depends on how you want your application to behave when something goes wrong.

### Quick answer

- Use `TaskGroup` when requests should succeed or fail together, and you want safer, structured error handling.
- Use `asyncio.gather(..., return_exceptions=True)` when you want more flexible behavior, especially if you need to keep partial successes even when some requests fail.

### How they differ in practice

| Area | `asyncio.gather` | `asyncio.TaskGroup` |
|---|---|---|
| Main idea | Run many awaitables and collect results | Run many tasks in one managed group |
| Result handling | Returns a result list directly | Keep task references and call `.result()` |
| Error behavior | Configurable with `return_exceptions` | Fail-fast by design |
| Partial success support | Strong, especially with `return_exceptions=True` | Limited, because one failure cancels remaining tasks |
| Safety and structure | Flexible but easier to misuse in complex flows | Strong structured concurrency, easier to reason about |
| Best for | Mixed outcomes, reporting all responses | All-or-nothing operations |

### Tradeoffs

#### `asyncio.gather` advantages

- Simple for batching and collecting results in one line.
- With `return_exceptions=True`, you can process successful and failed responses together.
- Useful when your app should continue even if some instruments fail.

#### `asyncio.gather` limitations

- Error behavior depends on options and can be confusing in larger workflows.
- Without careful handling, failed tasks can make control flow harder to follow.

#### `TaskGroup` advantages

- Clear lifecycle: tasks are created, managed, and completed within one block.
- Fail-fast behavior is safer for operations that must stay consistent.
- Better for production flows where cleanup and predictable control flow matter.

#### `TaskGroup` limitations

- More verbose: you must keep task references and extract results manually.
- Not ideal when partial results are the goal, because one failure cancels sibling tasks.

### Performance vs Asyncio.Gather

In practice, `asyncio.TaskGroup` can show *a slight performance advantage* over `asyncio.gather` in some Python 3.11+ workloads, partly due to improvements in task management and structured concurrency. However, the difference are micro-level and irrelevant.

That said, performance depends on workload characteristics and runtime conditions, so results may vary. Please check a direct comparison examples using Data Library Historical Pricing `historical_pricing.summaries.Definition` definition with 30 RICs on [30 RICs with TaskGroup](./ld_notebook_taskgroup_performance.ipynb) and [30 RICs with Gather](./ld_notebook_gather_performance.ipynb).

### Recommended choice for Data Library scenarios

Choose `asyncio.gather(..., return_exceptions=True)` when:

1. You are exploring data from many RICs.
2. You want to keep successful responses even if some requests fail.
3. You need one combined list of successes and errors for reporting.

Choose `TaskGroup` when:

1. You are running a strict batch job.
2. The whole operation should stop if any request fails.
3. You want clearer, safer concurrency structure.

### Practical rule of thumb

- If you want best-effort results: use `asyncio.gather(..., return_exceptions=True)`.
- If you want all-or-nothing correctness: use `asyncio.TaskGroup`.
- For both approaches, if you are sending many requests at once: add `asyncio.Semaphore(...)` to limit in-flight requests and reduce the chance of HTTP 429 errors.